![MangaJaNai Colab](https://raw.githubusercontent.com/0xH4KU/mangajanai-colab/main/assets/banner.svg)

# MangaJaNai Colab

**Sharper manga and color illustrations, powered by a Colab GPU.**

| Input | Models | Output |
| :--- | :--- | :--- |
| PDF, CBZ, ZIP, image folders | MangaJaNai + IllustrationJaNai · 2x FP16 | Lossless PNG pages in verified ZIPs |

Run the six numbered cells in order. Start with the preview, check the images, then process the full batch.
Original files stay in place. Each finished page is checkpointed to Drive, so interrupted books can resume.

### Before you start

1. Select **Runtime → Change runtime type → GPU**.
2. Put your books in **Google Drive → My Drive → Manga → input**, or set your own input folder below.
3. Use separate input and output folders. Allow space for model caches, PNG checkpoints, and finished ZIPs.

**Defaults:** 1920p for manga, V3denoise FDAT for color pages, full-page inference, and checkpoints kept.
This is an independent community notebook using models by [the-database](https://github.com/the-database/MangaJaNai).


In [ ]:
# @title 1. Choose folders, models, and output settings
# @markdown ### Google Drive folders
INPUT_DIR = "/content/drive/MyDrive/Manga/input" # @param {type:"string"}
OUTPUT_DIR = "/content/drive/MyDrive/Manga/output" # @param {type:"string"}
# @markdown ### Black-and-white pages
MODEL_SELECTION = "Auto" # @param ["Auto", "1200p", "1300p", "1400p", "1500p", "1600p", "1920p", "2048p"]
# @markdown Optional official manga model file. Leave empty for **Auto** or automatic downloads.
MODEL_PATH = "" # @param {type:"string"}
# @markdown ### Color pages and covers
ILLUSTRATION_SELECTION = "V3denoise FDAT" # @param ["V3denoise FDAT", "V3detail FDAT", "V3denoise SPAN", "V3detail SPAN", "Keep original"]
# @markdown ### Processing and storage
# @markdown **0** processes a full page. Smaller tiles reduce VRAM use.
TILE_SIZE = 0 # @param [0, 512, 384, 256] {type:"raw"}
# @markdown Keep resumable PNGs. Uncheck to remove them only after the completed ZIP is verified.
KEEP_PAGE_CHECKPOINTS = True # @param {type:"boolean"}


In [ ]:
# @title 2. Connect the GPU and Google Drive
import subprocess
import sys
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU, then run this cell again.")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "spandrel==0.4.2", "pymupdf==1.28.2", "pillow", "tqdm",
])

from google.colab import drive
drive.mount("/content/drive")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | VRAM: {gpu.total_memory / 2**30:.2f} GiB")


In [ ]:
# @title 3. Load the inference engine
import hashlib
import io
import json
import math
import re
import shutil
import tempfile
import time
import urllib.request
from contextlib import closing
from pathlib import Path, PurePosixPath
from zipfile import ZipFile, ZIP_STORED

import numpy as np
import pymupdf
import torch
from PIL import Image, ImageOps, PngImagePlugin
from spandrel import ImageModelDescriptor, ModelLoader
from tqdm.auto import tqdm

MODEL_URL = "https://github.com/the-database/MangaJaNai/releases/download/1.0.0/MangaJaNai_V1_ModelsOnly.zip"
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
ARCHIVE_SUFFIXES = {".cbz", ".zip"}
PDF_RENDER_HEIGHT = 1920
PAD = 32
MODELS = {
    1200: ('2x_MangaJaNai_1200p_V1_ESRGAN_70k.pth', '43b784f674bdbf89886a62a64cd5f8d8df92caf4d861bdf4d47dad249ede0267'),
    1300: ('2x_MangaJaNai_1300p_V1_ESRGAN_75k.pth', '15ca3c0f75f97f7bf52065bf7c9b8d602de94ce9e3b078ac58793855eed18589'),
    1400: ('2x_MangaJaNai_1400p_V1_ESRGAN_70k.pth', 'a940ad8ebcf6bea5580f2f59df67deb009f054c9b87dbbc58c2e452722f34858'),
    1500: ('2x_MangaJaNai_1500p_V1_ESRGAN_90k.pth', 'd91f2d247fa61144c1634a2ba46926acd3956ae90d281a5bed6655f8364a5b2c'),
    1600: ('2x_MangaJaNai_1600p_V1_ESRGAN_90k.pth', '6f5923f812dbc5d6aeed727635a21e74cacddce595afe6135cbd95078f6eee44'),
    1920: ('2x_MangaJaNai_1920p_V1_ESRGAN_70k.pth', '1ad4aa6f64684baa430da1bb472489bff2a02473b14859015884a3852339c005'),
    2048: ('2x_MangaJaNai_2048p_V1_ESRGAN_95k.pth', '146cd009b9589203a8444fe0aa7195709bb5b9fdeaca3808b7fbbd5538f94c41'),
}

ILLUSTRATION_MODELS = {
    'V3detail FDAT': ('2x_IllustrationJaNai_V3detail_FDAT_M_unshuffle_40k_fp16.safetensors', '8ff2ba0b970e6939f3711077f5260fcded16b8388a939a27f153c687ff5da3e0', 'detail'),
    'V3detail SPAN': ('2x_IllustrationJaNai_V3detail_SPAN_S_40k_fp16.safetensors', '6b10bd97260ebcf639db9a961121a22c97c4f8807d43ec8a1157f3ef24b18f46', 'detail'),
    'V3denoise FDAT': ('2x_IllustrationJaNai_V3denoise_FDAT_M_unshuffle_30k_fp16.safetensors', '960c9555f62fb19113e8d808bde5a54b64181b1db417bca1bbda26516e48f108', 'denoise'),
    'V3denoise SPAN': ('2x_IllustrationJaNai_V3denoise_SPAN_S_30k_fp16.safetensors', 'd709bfb8f7cefcd0bc0d2dce485208dcda165b2559eacec66296b53d486f0951', 'denoise'),
}

ILLUSTRATION_PACKAGES = {
    "detail": ("https://github.com/the-database/MangaJaNai/releases/download/3.0.0/IllustrationJaNai_V3detail.zip",
               "27749962bc934779f867730cdb10617e3d7475313f920a7a966c56b75dbdbeda"),
    "denoise": ("https://github.com/the-database/MangaJaNai/releases/download/3.0.0/IllustrationJaNai_V3denoise.zip",
                "6c26828ad8e142c9e9c7181451257f3f56a0f5c7f897d9c7878ce898404f53e5"),
}


def model_spec(key):
    return MODELS[key] if key in MODELS else ILLUSTRATION_MODELS[key][:2]


def natural_key(name):
    return [int(p) if i % 2 else p.casefold()
            for i, p in enumerate(re.split(r"(\d+)", str(name)))]

def safe_member(name):
    path = PurePosixPath(name.replace("\\", "/"))
    if path.is_absolute() or ".." in path.parts or not path.parts or ":" in path.parts[0]:
        raise ValueError(f"Unsafe archive path: {name!r}")
    return path.as_posix()

def is_auxiliary(name):
    path = PurePosixPath(name)
    return "__MACOSX" in path.parts or path.name == ".DS_Store" or path.name.startswith("._")


def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def choose_model(selection, height):
    if selection == "Auto":
        return min(MODELS, key=lambda target: (abs(target - height), target))
    if selection not in {f"{height}p" for height in MODELS}:
        raise ValueError("Choose Auto or a supported manga model height.")
    return int(selection[:-1])


class ModelPool:
    def __init__(self):
        self.selection = MODEL_SELECTION
        self.illustration_selection = ILLUSTRATION_SELECTION
        self.model_path = MODEL_PATH.strip()
        choose_model(self.selection, 1920)
        if self.illustration_selection not in ILLUSTRATION_MODELS and self.illustration_selection != "Keep original":
            raise ValueError("Choose an IllustrationJaNai model or Keep original.")
        if self.model_path:
            if self.selection == "Auto":
                raise ValueError("Leave MODEL_PATH empty in Auto mode; models are selected from the cache.")
            if not Path(self.model_path).is_file():
                raise FileNotFoundError(f"Model not found: {self.model_path}")
            expected = MODELS[choose_model(self.selection, 1920)][1]
            if file_hash(self.model_path) != expected:
                raise ValueError("MODEL_PATH must match the selected official manga model.")
        self.cache = Path(OUTPUT_DIR) / "models"
        self.active = None
        self.model = None

    def get(self, key):
        if self.active == key:
            return self.model
        name, expected = model_spec(key)
        path = Path(self.model_path) if self.model_path and key in MODELS else self.cache / name
        if not path.exists():
            self.cache.mkdir(parents=True, exist_ok=True)
            if key in MODELS:
                url, package_hash, weights_to_cache = MODEL_URL, None, list(MODELS.values())
            else:
                variant = ILLUSTRATION_MODELS[key][2]
                url, package_hash = ILLUSTRATION_PACKAGES[variant]
                weights_to_cache = [spec[:2] for spec in ILLUSTRATION_MODELS.values() if spec[2] == variant]
            print(f"Downloading and caching official weights: {name}...")
            with tempfile.TemporaryDirectory() as work:
                archive_path = Path(work) / "models.zip"
                with urllib.request.urlopen(url, timeout=60) as src, archive_path.open("wb") as dst:
                    shutil.copyfileobj(src, dst)
                if package_hash and file_hash(archive_path) != package_hash:
                    raise ValueError("IllustrationJaNai package checksum mismatch.")
                with ZipFile(archive_path) as archive:
                    for filename, sha in weights_to_cache:
                        weights = archive.read(filename)
                        if hashlib.sha256(weights).hexdigest() != sha:
                            raise ValueError(f"Official model checksum mismatch: {filename}")
                        target = self.cache / filename
                        partial = target.with_name(target.name + ".part")
                        partial.write_bytes(weights)
                        partial.replace(target)
        if file_hash(path) != expected:
            raise ValueError(f"Model checksum mismatch: {path}; use the original official model for this version.")
        self.model = None
        self.active = None
        torch.cuda.empty_cache()
        model = ModelLoader().load_from_file(path)
        if not isinstance(model, ImageModelDescriptor) or model.scale != 2 or not model.supports_half:
            raise ValueError("The model must support 2x image upscaling and FP16.")
        self.model = model.eval().half().cuda()
        self.active = key
        print(f"Loading {name}; FP16; device {self.model.device}")
        return self.model


def load_model():
    pool = ModelPool()
    print(f"Manga model: {pool.selection}; color model: {pool.illustration_selection}; loaded on demand. Cache: {pool.cache}")
    return pool


def read_page(raw):
    with Image.open(io.BytesIO(raw)) as source:
        if getattr(source, "n_frames", 1) != 1:
            raise ValueError("Animated or multipage images are unsupported. Convert them to individual images first.")
        image = ImageOps.exif_transpose(source)
        if image.mode in {"I", "F"} or image.mode.startswith("I;16"):
            raise ValueError("Convert high-bit-depth images to 8-bit first to avoid clipped grayscale values.")
        if image.mode in {"RGBA", "LA"} or "transparency" in image.info:
            rgba = image.convert("RGBA")
            background = Image.new("RGBA", rgba.size, "white")
            image = Image.alpha_composite(background, rgba)
        return image.convert("RGB")

def is_color(image):
    # shortcut: tinted paper may count as color; add a manual override for scanned books if needed.
    sample = image.copy()
    sample.thumbnail((256, 256))
    rgb = np.asarray(sample, dtype=np.int16)
    return np.percentile(rgb.max(axis=2) - rgb.min(axis=2), 99) > 12

@torch.inference_mode()
def infer(image, model, tile):
    rgb = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    height, width, _ = rgb.shape
    step = tile or max(height, width)
    output = np.empty((height * 2, width * 2, 3), dtype=np.uint8)
    for y in range(0, height, step):
        for x in range(0, width, step):
            bottom, right = min(y + step, height), min(x + step, width)
            top_pad, left_pad = max(y - PAD, 0), max(x - PAD, 0)
            bottom_pad, right_pad = min(bottom + PAD, height), min(right + PAD, width)
            patch = rgb[top_pad:bottom_pad, left_pad:right_pad]
            tensor = torch.from_numpy(patch).permute(2, 0, 1).unsqueeze(0)
            tensor = tensor.to(device=model.device, dtype=model.dtype)
            prediction = model(tensor).squeeze(0).permute(1, 2, 0).cpu().float().numpy()
            if not np.isfinite(prediction).all():
                raise FloatingPointError("FP16 inference produced NaN/Inf. Stopping before saving an invalid image.")
            prediction = np.rint(np.clip(prediction, 0, 1) * 255).astype(np.uint8)
            crop_y, crop_x = (y - top_pad) * 2, (x - left_pad) * 2
            output[y * 2:bottom * 2, x * 2:right * 2] = prediction[
                crop_y:crop_y + (bottom - y) * 2, crop_x:crop_x + (right - x) * 2
            ]
    result = Image.fromarray(output)
    return result.convert("L") if image.mode == "L" else result

def upscale(image, model, tile):
    while True:
        try:
            return infer(image, model, tile), tile
        except torch.cuda.OutOfMemoryError:
            if tile and tile <= 128:
                raise RuntimeError("Insufficient VRAM even with 128-pixel tiles. Restart the GPU runtime.") from None
            tile = 512 if tile == 0 else max(128, tile // 8 * 4)
            print(f"Out of VRAM. Retrying this page with tile size {tile}")
        # Leaving except releases tensors held by the failed inference traceback.
        torch.cuda.empty_cache()

def valid_result(path, signature):
    try:
        with Image.open(path) as image:
            matches = image.info.get("mangajanai_signature") == signature
            image.verify()
            return matches
    except (OSError, SyntaxError, ValueError):
        return False

def pdf_page_bytes(page):
    scales = []
    for image in page.get_image_info():
        rect = pymupdf.Rect(image["bbox"]) * page.rotation_matrix
        coverage = (rect & page.rect).get_area() / page.rect.get_area()
        if coverage < 0.9 or min(image["width"], image["height"]) < 4:
            continue
        matrix = pymupdf.Matrix(image["transform"])
        width, height = math.hypot(matrix.a, matrix.b), math.hypot(matrix.c, matrix.d)
        if width and height:
            scales.append(max(image["width"] / width, image["height"] / height))
    # shortcut: composite/vector pages render at 1920p; raise PDF_RENDER_HEIGHT when more detail is needed.
    scale = max(scales) if scales else PDF_RENDER_HEIGHT / page.rect.height
    pixmap = page.get_pixmap(matrix=pymupdf.Matrix(scale, scale), colorspace=pymupdf.csRGB, alpha=False)
    return pixmap.tobytes("png")


def process_page(raw, output, local_output, pool, tiles, stats):
    started = time.perf_counter()
    image = read_page(raw)
    color = is_color(image)
    color_upscale = color and pool.illustration_selection != "Keep original"
    height = choose_model(pool.selection, image.height)
    key = pool.illustration_selection if color_upscale else height
    name, sha = model_spec(key)
    # Preserve existing grayscale/original-page signatures; RGB upscales use a separate signature.
    settings = {"version": 1, "model": sha, "tile": TILE_SIZE, "pad": PAD}
    if color_upscale:
        settings.update(version=2, rgb=True)
    signature = hashlib.sha256(json.dumps(settings).encode() + raw).hexdigest()
    stats["prepare"] += time.perf_counter() - started
    local_output.parent.mkdir(parents=True, exist_ok=True)
    tile = tiles.get(key, TILE_SIZE)
    kind = "Original color page" if color and not color_upscale else name
    started = time.perf_counter()
    if valid_result(output, signature):
        shutil.copyfile(output, local_output)
        stats["drive"] += time.perf_counter() - started
        print(f"Reusing: {output.name}; {kind}")
        return tile, True
    stats["drive"] += time.perf_counter() - started
    allocated = reserved = inference_seconds = 0.0
    if color and not color_upscale:
        result = image
    else:
        started = time.perf_counter()
        model = pool.get(key)
        stats["model"] += time.perf_counter() - started
        cuda = model.device.type == "cuda"
        if cuda:
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        result, tile = upscale(image if color else image.convert("L"), model, tile)
        tiles[key] = tile
        if cuda:
            torch.cuda.synchronize()
            allocated = torch.cuda.max_memory_allocated() / 2**30
            reserved = torch.cuda.max_memory_reserved() / 2**30
        inference_seconds = time.perf_counter() - started
        stats["infer"] += inference_seconds
    started = time.perf_counter()
    metadata = PngImagePlugin.PngInfo()
    metadata.add_text("mangajanai_signature", signature)
    result.save(local_output, format="PNG", pnginfo=metadata, compress_level=3)
    stats["encode"] += time.perf_counter() - started
    started = time.perf_counter()
    output.parent.mkdir(parents=True, exist_ok=True)
    partial = output.with_name(output.name + ".part")
    shutil.copyfile(local_output, partial)
    partial.replace(output)
    stats["drive"] += time.perf_counter() - started
    print(f"{output.name}: {image.size} -> {result.size}; {kind}; tile {tile or 'full page'}; inference {inference_seconds:.2f} s; peak allocated {allocated:.2f}/reserved {reserved:.2f} GiB")
    return tile, False


def prepare_book(source, root, work):
    if isinstance(source, list):
        inputs, hashes = [], []
        for path in source:
            name = path.relative_to(root).as_posix()
            target = work / "input" / name
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(path, target)
            inputs.append((target, name + ".png"))
            hashes.append((name, file_hash(target)))
        digest = hashlib.sha256(json.dumps(hashes, ensure_ascii=False).encode()).hexdigest()
        return inputs, digest
    target = work / ("input.pdf" if source.suffix.lower() == ".pdf" else "input.zip")
    shutil.copyfile(source, target)
    return target, file_hash(target)


def book_pages(source, stats):
    if isinstance(source, list):
        for path, name in source:
            started = time.perf_counter()
            raw = path.read_bytes()
            stats["prepare"] += time.perf_counter() - started
            yield name, raw
        return
    if source.suffix == ".pdf":
        with pymupdf.open(source) as document:
            if document.needs_pass:
                raise ValueError("This PDF is password protected. Unlock it first.")
            if not len(document):
                raise ValueError("The PDF has no pages.")
            for index in range(len(document)):
                started = time.perf_counter()
                raw = pdf_page_bytes(document[index])
                stats["render"] += time.perf_counter() - started
                yield f"{index + 1:05d}.png", raw
        return
    with ZipFile(source) as archive:
        members, seen = [], set()
        for entry in archive.infolist():
            name = safe_member(entry.filename)
            if entry.is_dir() or is_auxiliary(name):
                continue
            if name in seen:
                raise ValueError(f"Duplicate archive filename: {name}")
            seen.add(name)
            if PurePosixPath(name).suffix.lower() in IMAGE_SUFFIXES:
                members.append((entry, name))
        if not members:
            raise ValueError("The archive contains no supported images.")
        members.sort(key=lambda member: natural_key(member[1]))
        for entry, name in members:
            started = time.perf_counter()
            raw = archive.read(entry)
            stats["prepare"] += time.perf_counter() - started
            yield name + ".png", raw


def completion_path(book):
    return book.with_name(book.name + ".done.json")


def completed_book(book, signature):
    try:
        record = json.loads(completion_path(book).read_text())
        if not isinstance(record, dict) or not isinstance(record.get("pages"), list):
            return None
        if record["signature"] != signature or record["zip_bytes"] != book.stat().st_size:
            return None
        if file_hash(book) != record["zip_sha256"]:
            return None
        return record
    except (OSError, ValueError, KeyError, TypeError):
        return None


def clean_checkpoints(page_root, names, destination):
    for name in names:
        path = page_root / safe_member(name)
        if not path.resolve().is_relative_to(destination) or path.is_symlink():
            raise ValueError(f"Refusing to clean files outside the output folder: {path}")
        if path.is_file():
            try:
                with Image.open(path) as image:
                    owned = re.fullmatch(r"[0-9a-f]{64}", image.info.get("mangajanai_signature", ""))
            except (OSError, SyntaxError, ValueError):
                continue
            if owned:
                path.unlink()


def pack_zip(book, pages, stats, archive=None):
    started = time.perf_counter()
    with tempfile.TemporaryDirectory() as work:
        local_result = Path(work) / "result.zip"
        with ZipFile(local_result, "w", compression=ZIP_STORED) as result:
            for page, name in pages:
                result.write(page, arcname=name)
            if archive is not None:
                with ZipFile(archive) as source:
                    for entry in source.infolist():
                        name = safe_member(entry.filename)
                        if entry.is_dir() or is_auxiliary(name) or PurePosixPath(name).suffix.lower() in IMAGE_SUFFIXES:
                            continue
                        with source.open(entry) as src, result.open(name, "w", force_zip64=True) as dst:
                            shutil.copyfileobj(src, dst)
        with ZipFile(local_result) as result:
            if result.testzip() is not None:
                raise OSError("Local ZIP verification failed. The previous output has been preserved.")
        expected = file_hash(local_result)
        size = local_result.stat().st_size
        stats["pack"] += time.perf_counter() - started
        started = time.perf_counter()
        book.parent.mkdir(parents=True, exist_ok=True)
        partial = book.with_name(book.name + ".part")
        shutil.copyfile(local_result, partial)
        if file_hash(partial) != expected:
            raise OSError("ZIP upload verification failed. The previous output has been preserved.")
        partial.replace(book)
        stats["upload"] += time.perf_counter() - started
    print(f"ZIP saved and verified: {book}")
    return expected, size


def report_times(stats):
    labels = {"prepare": "Input preparation", "render": "PDF rendering", "model": "Model loading",
              "infer": "GPU inference", "encode": "PNG encoding", "drive": "Drive checkpoints",
              "pack": "Local ZIP packing and verification", "upload": "ZIP upload and verification"}
    print(" | ".join(f"{label} {stats[key]:.2f} s" for key, label in labels.items()))


def run_batch(pool, limit=None):
    root, destination = Path(INPUT_DIR).resolve(), Path(OUTPUT_DIR).resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Input folder not found: {root}")
    if root == destination or root in destination.parents or destination in root.parents:
        raise ValueError("Input and output must be separate folders; neither may contain the other.")
    if type(TILE_SIZE) is not int or TILE_SIZE < 0 or TILE_SIZE % 4:
        raise ValueError("TILE_SIZE must be 0 or a positive multiple of 4.")
    if type(KEEP_PAGE_CHECKPOINTS) is not bool:
        raise ValueError("KEEP_PAGE_CHECKPOINTS must be True or False.")
    if pool.selection != MODEL_SELECTION or pool.illustration_selection != ILLUSTRATION_SELECTION or pool.model_path != MODEL_PATH.strip() or pool.cache.resolve() != destination / "models":
        raise ValueError("Models or output folder changed. Run step 4 again.")
    sources = sorted(
        (p for p in root.rglob("*") if p.is_file()
         and p.suffix.lower() in IMAGE_SUFFIXES | ARCHIVE_SUFFIXES | {".pdf"}
         and not is_auxiliary(p.relative_to(root).as_posix())), key=natural_key
    )
    if not sources:
        raise ValueError("No JPG, PNG, WebP, BMP, PDF, CBZ, or ZIP files found in the input folder.")
    images = [p for p in sources if p.suffix.lower() in IMAGE_SUFFIXES]
    jobs, images_added = [], False
    for source in sources:
        if source.suffix.lower() in IMAGE_SUFFIXES:
            if not images_added:
                jobs.append((images, destination / "books" / "images.2x.zip", destination / "pages" / "_images"))
                images_added = True
        else:
            relative = source.relative_to(root)
            kind = "_pdfs" if source.suffix.lower() == ".pdf" else "_archives"
            jobs.append((source, destination / "books" / (str(relative) + ".2x.zip"), destination / "pages" / kind / relative))
    settings = {"version": 1, "selection": pool.selection, "models": MODELS,
                "illustration": pool.illustration_selection, "illustration_models": ILLUSTRATION_MODELS,
                "tile": TILE_SIZE, "pad": PAD, "pdf_height": PDF_RENDER_HEIGHT}
    started_batch = time.perf_counter()
    tiles = {}
    visited, skipped, completed = 0, 0, 0
    for source, book, page_root in tqdm(jobs, desc="Books"):
        stats = {key: 0.0 for key in ("prepare", "render", "model", "infer", "encode", "drive", "pack", "upload")}
        with tempfile.TemporaryDirectory() as work:
            work = Path(work)
            started = time.perf_counter()
            local_input, digest = prepare_book(source, root, work)
            stats["prepare"] += time.perf_counter() - started
            signature = hashlib.sha256(json.dumps({"source": digest, "settings": settings}).encode()).hexdigest()
            if limit is None:
                started = time.perf_counter()
                record = completed_book(book, signature)
                stats["drive"] += time.perf_counter() - started
                if record is not None:
                    print(f"Book complete; ZIP checksum verified: {book.name}")
                    completed += 1
                    if not KEEP_PAGE_CHECKPOINTS:
                        clean_checkpoints(page_root, record["pages"], destination)
                    report_times(stats)
                    continue
            outputs = []
            with closing(book_pages(local_input, stats)) as pages:
                for name, raw in tqdm(pages, desc=book.name, leave=False):
                    local_output = work / "pages" / name
                    _, was_skipped = process_page(raw, page_root / name, local_output, pool, tiles, stats)
                    outputs.append((local_output, name))
                    visited += 1
                    skipped += was_skipped
                    if limit is not None and visited >= limit:
                        print(f"Preview finished: {visited} pages ({skipped} reused pages). Run the batch cell to continue.")
                        report_times(stats)
                        return
            archive = local_input if isinstance(local_input, Path) and local_input.suffix == ".zip" else None
            sha, size = pack_zip(book, outputs, stats, archive)
            record = {"signature": signature, "zip_sha256": sha, "zip_bytes": size,
                      "pages": [name for _, name in outputs], "seconds": stats}
            marker = completion_path(book)
            partial = marker.with_name(marker.name + ".part")
            partial.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
            partial.replace(marker)
            if not KEEP_PAGE_CHECKPOINTS and limit is None:
                clean_checkpoints(page_root, record["pages"], destination)
            report_times(stats)
    print(f"Done: {visited} pages ({skipped} reused pages); {completed} complete books skipped; elapsed {time.perf_counter() - started_batch:.1f} s.")


In [ ]:
# @title 4. Prepare model selection and the cache
model = load_model()


## Preview before you batch

Step 5 saves the first three pages to Drive. Check linework, text, halftones, color, and the reported model.
The first inference can include initialization time. The summary separates reading, PDF rendering, model loading,
GPU inference, PNG encoding, checkpoint transfers, ZIP packing, and upload verification.

| Setting | What it does |
| :--- | :--- |
| **Manga · Auto** | Matches each full input page's height to 1200p–2048p; midpoint ties use the smaller model. |
| **Manga · a fixed height** | Uses that model for every detected grayscale page. The number is an input-height target, not a quality ranking. |
| **Color · denoise / detail** | Denoise cleans halftone noise; detail favors texture. Try both on a representative cover. |
| **Color · FDAT / SPAN** | Both run at 2x FP16. SPAN is an option for lightly degraded images. |
| **Color · Keep original** | Keeps the color page at its input size. |

After changing a model or output folder, rerun **steps 1 and 4**, then preview.
After changing only the tile size, rerun **step 1**, then preview. A changed setting can replace previous outputs at the same paths.

Preview checks individual pages even when a completed ZIP exists. The batch step can skip whole completed books.


In [ ]:
# @title 5. Preview the first 3 pages
run_batch(model, limit=3)


In [ ]:
# @title 6. Process the batch and save verified ZIPs
run_batch(model)


## Resume and find your results

Reconnect to a GPU, mount the same Drive, keep the same settings, and run **steps 1–4, then step 6**.
Matching page checkpoints are reused. Compatible grayscale checkpoints from earlier 1920p versions also work.
Changing the color model recalculates color pages while matching grayscale pages are reused.

```text
output/
├── books/    Finished ZIPs and their .done.json completion records
├── pages/    Resumable PNG checkpoints
└── models/   Verified official weights cached across sessions
```

Download the ZIP to read your book. Keep its `.done.json` beside it in Drive for future batch runs.
Whole-book skipping verifies source, settings, and the ZIP hash; it still reads the files, but avoids rendering,
inference, and repacking. Missing or mismatched records fall back to page checks. Partial `.part` files never count as complete.

### Storage

Each page is encoded locally, then checkpointed to Drive. ZIPs are packed from local files.
The Colab disk needs room for the current book's input, output PNGs, and ZIP.
Drive stores both PNGs and ZIPs when checkpoints are kept.

Uncheck **KEEP_PAGE_CHECKPOINTS** to remove this tool's page PNGs only after the ZIP is verified and its completion record is saved.
Unfinished books and previews retain checkpoints. If a ZIP is later lost after cleanup, that book must be upscaled again.
Source files, model caches, and unrelated files are kept.

### Downloads and compatibility

- Manga models: a **433 MB** package on first use; about **236 MB** of 2x weights stay in the cache.
- Illustration models: **V3denoise · 167 MB** or **V3detail · 319 MB** on demand; about **25 MB** of 2x weights per family are cached.
- One model is active on the GPU at a time. Out-of-memory tile reductions are remembered independently per model.
- PDFs render their full visible page, including rotation, crop, text, and annotations. Full-page scans use native pixel density;
  composite/vector pages default to 1920 pixels high. Adjust `PDF_RENDER_HEIGHT` in step 3 if needed.
- Supported raster inputs: single-frame **8-bit JPG/JPEG, PNG, WebP, BMP**. Transparent pixels are composited on white.
  Convert high-bit-depth grayscale images first. Unlock encrypted PDFs before processing.
- Archive metadata such as `ComicInfo.xml` is retained. Page names add `.png` to prevent JPG/PNG filename collisions.
  PDF pages use `00001.png` numbering. Loose images share `images.2x.zip`, with their folder structure retained.

### Practical limits

Color detection uses channel differences; tinted paper can be classified as color.
Tiles use 32 pixels of context and can differ slightly from full-page inference.
IllustrationJaNai V3 is an upstream prerelease. Compare detail and denoise before processing many books.
This notebook supports **2x**; 4x, CBR/RAR, and EPUB conversion are outside its scope.
[Colab resources vary](https://research.google.com/colaboratory/faq.html); GPU speed should be measured on your assigned runtime.

---

**Project:** [GitHub](https://github.com/0xH4KU/mangajanai-colab) · [Source license](https://github.com/0xH4KU/mangajanai-colab/blob/main/LICENSE) · [Third-party notices](https://github.com/0xH4KU/mangajanai-colab/blob/main/THIRD_PARTY_NOTICES.md)

Copyright © 2026 HAKU. Original project code is licensed under **AGPL-3.0-only**, without warranty.
Model weights retain their upstream **CC BY-NC 4.0** terms and are downloaded from official releases, not bundled here.
Thanks to [the-database](https://github.com/the-database/MangaJaNai), [Spandrel](https://github.com/chaiNNer-org/spandrel), and [PyMuPDF](https://pymupdf.readthedocs.io/).
